# Using Anthropic (Claude) as Your Model Provider

This notebook shows how to use Anthropic's Claude models with MemoRizz agents.
Claude excels at nuanced reasoning, code generation, and following complex instructions.

**What you'll learn:**
1. Setting up the Anthropic provider
2. Using different Claude models (Opus, Sonnet, Haiku)
3. Tool calling with Claude
4. Streaming responses
5. How Claude's API differs from OpenAI (system messages, content blocks)

> **Prerequisites:** An Anthropic API key. Get one at [console.anthropic.com](https://console.anthropic.com)

In [ ]:
%pip install -qU memorizz anthropic

In [ ]:
import os
import getpass

if not os.getenv("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("API key configured.")

---
## Method 1: Config Dict (Recommended)

The simplest approach — just change `"provider"` to `"anthropic"` in your config dict.

In [ ]:
from memorizz.memagent.builders import MemAgentBuilder

agent = (
    MemAgentBuilder()
    .with_instruction(
        "You are a helpful assistant that explains concepts clearly and concisely."
    )
    .with_llm_config({
        "provider": "anthropic",
        "model": "claude-sonnet-4-5-20250929",
    })
    .build()
)

response = agent.run("What is the difference between concurrency and parallelism?")
print(response)

## Method 2: Direct Provider Instance

For fine-grained control over parameters:

In [ ]:
from memorizz.llms import Anthropic
from memorizz.memagent.builders import MemAgentBuilder

llm = Anthropic(
    model="claude-sonnet-4-5-20250929",
    temperature=0.3,
    max_tokens=2048,
    top_k=40,
)

agent = (
    MemAgentBuilder()
    .with_instruction("You are a Python expert. Give concise, production-quality code.")
    .with_model(llm)
    .build()
)

response = agent.run("Write a Python function that implements binary search.")
print(response)

---
## Choosing the Right Claude Model

Anthropic offers three tiers, each balancing capability and cost:

| Model | Best For | Context | Speed |
|-------|----------|---------|-------|
| **Claude Opus 4.6** | Complex reasoning, research, multi-step analysis | 200K | Slower |
| **Claude Sonnet 4.5** | Best balance of quality, speed, and cost | 200K | Medium |
| **Claude Haiku 4.5** | Fast tasks, classification, extraction | 200K | Fastest |

All Claude models support 200K token context windows — larger than most competitors.

In [ ]:
# Quick comparison: same prompt, different models
for model_id in ["claude-haiku-4-5-20251001", "claude-sonnet-4-5-20250929"]:
    quick_agent = (
        MemAgentBuilder()
        .with_instruction("Be brief.")
        .with_llm_config({"provider": "anthropic", "model": model_id})
        .build()
    )
    resp = quick_agent.run("In one sentence, what is a transformer in ML?")
    print(f"\n{model_id}:\n{resp}")

---
## Tool Calling with Claude

Claude supports tool calling (function calling). MemoRizz handles the format conversion
automatically — you register tools the same way as with OpenAI.

In [ ]:
from memorizz.memagent.builders import MemAgentBuilder


def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculate Body Mass Index given weight in kg and height in meters."""
    bmi = weight_kg / (height_m ** 2)
    category = (
        "underweight" if bmi < 18.5
        else "normal" if bmi < 25
        else "overweight" if bmi < 30
        else "obese"
    )
    return f"BMI: {bmi:.1f} ({category})"


agent = (
    MemAgentBuilder()
    .with_instruction("You are a health assistant. Use tools when calculations are needed.")
    .with_llm_config({"provider": "anthropic", "model": "claude-sonnet-4-5-20250929"})
    .build()
)

agent.register_tool(calculate_bmi)

response = agent.run("I weigh 75 kg and I'm 1.78 m tall. What's my BMI?")
print(response)

---
## Streaming Responses

The Anthropic provider supports streaming via the same `run_stream()` interface.

In [ ]:
stream_agent = (
    MemAgentBuilder()
    .with_instruction("You are a storyteller.")
    .with_llm_config({"provider": "anthropic", "model": "claude-sonnet-4-5-20250929"})
    .build()
)

for event in stream_agent.run_stream("Write a very short story about a robot who learns to paint."):
    if event.get("type") == "content":
        print(event["content"], end="", flush=True)
print()

---
## Key Differences from OpenAI

MemoRizz abstracts these differences, but it's useful to know:

| Aspect | OpenAI | Anthropic (Claude) |
|--------|--------|--------------------|
| System prompt | Message with `role: system` | Top-level `system` parameter |
| Response format | `choices[0].message.content` | `content[0].text` (content blocks) |
| Tool definitions | `parameters` key | `input_schema` key |
| Tool choice | `"auto"`, `"none"`, specific | `{"type": "auto"}`, `{"type": "none"}`, etc. |
| Max tokens | Optional (has defaults) | **Required** parameter |
| Context window | Varies (8K-128K) | 200K for all Claude 3+ models |

The MemoRizz Anthropic provider handles all conversions automatically.

## Config Reference

| Key | Type | Default | Description |
|-----|------|---------|-------------|
| `provider` | str | — | Must be `"anthropic"` |
| `model` | str | `"claude-sonnet-4-5-20250929"` | Model ID |
| `api_key` | str | `ANTHROPIC_API_KEY` env var | API key |
| `temperature` | float | None | Sampling temperature (0.0-1.0) |
| `max_tokens` | int | 4096 | Maximum tokens to generate |
| `top_p` | float | None | Nucleus sampling |
| `top_k` | int | None | Top-K sampling |